Packaging design comparison showing original uniformity vs revival variety.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r packaging_data
SELECT
  era,
  product_name,
  product_type,
  packaging_theme,
  pkg_shape,
  pkg_has_chrome,
  pkg_has_novelty,
  pkg_has_mirror
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.PRODUCTS
ORDER BY era, pkg_shape, product_type;

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D

df = packaging_data.to_pandas() if hasattr(packaging_data, 'to_pandas') else packaging_data
df = df.copy()
df.columns = [col.lower() for col in df.columns]

revival_colors = {
    'Joystick Blush': '#FFD700',
    'Legally Bronze Bronzer': '#C0C0C0',
    'Drawn This Way Eyeliner': '#C0C0C0',
    'Born Star Eyeshadow': '#2E8B57',
    'Money Shot Highlighter': '#C0C0C0',
    'Heart On Lipstick': '#8B1A2C',
    'Flashes Mascara': '#B57EDC',
}

revival_markers = {
    'Joystick Blush': '*',
    'Legally Bronze Bronzer': 'D',
    'Drawn This Way Eyeliner': 'P',
    'Born Star Eyeshadow': '*',
    'Money Shot Highlighter': 'o',
    'Heart On Lipstick': 'h',
    'Flashes Mascara': '*',
}

shape_order = ['compact', 'tube', 'pencil', 'pen', 'bottle', 'other', 'stick', 'jar']
era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Uniform: all black, glossy, rounded pill shapes',
    'revival': 'Varied: colorful, playful shapes, novelty charms',
}

def plot_packaging(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14))
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = df[df['era'] == era].copy()
        era_df['shape_x'] = era_df['pkg_shape'].apply(lambda s: shape_order.index(s) if s in shape_order else len(shape_order))
        era_df = era_df.sort_values(['shape_x', 'product_type']).reset_index(drop=True)
        y_positions = []
        shape_counts_dict = {}
        for _, row in era_df.iterrows():
            shape = row['pkg_shape']
            shape_counts_dict[shape] = shape_counts_dict.get(shape, 0) + 1
            y_positions.append(shape_counts_dict[shape])
        era_df['y_pos'] = y_positions

        if era == 'original_launch':
            axis.scatter(
                era_df['shape_x'], era_df['y_pos'],
                s=200, c='black', edgecolors='#333333',
                linewidths=0.8, marker='o', alpha=0.9,
            )
        else:
            for _, row in era_df.iterrows():
                color = revival_colors.get(row['product_name'], '#C0C0C0')
                marker = revival_markers.get(row['product_name'], 'o')
                axis.scatter(
                    row['shape_x'], row['y_pos'],
                    s=280, c=color, edgecolors='black',
                    linewidths=0.8, marker=marker, alpha=0.9, zorder=5,
                )

        for _, row in era_df.iterrows():
            axis.annotate(
                row['product_name'].replace(' ', '\n', 1),
                (row['shape_x'], row['y_pos']),
                textcoords='offset points', xytext=(12, 0),
                fontsize=8, fontweight='normal', va='center', alpha=0.85,
            )

        shapes_in_era = sorted(era_df['pkg_shape'].unique(), key=lambda s: shape_order.index(s) if s in shape_order else len(shape_order))
        axis.set_xticks([shape_order.index(s) for s in shapes_in_era])
        axis.set_xticklabels(shapes_in_era, fontsize=9)
        axis.set_xlabel('Package shape', fontsize=11, fontweight='normal')
        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_ylim(0.3, max(era_df['y_pos']) + 1)
        axis.set_yticks([])
        axis.grid(axis='x', alpha=0.2)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
        axis.spines['left'].set_visible(False)
        axis.tick_params(labelsize=9)

    if layout == 'horizontal':
        fig.suptitle('ORIGINAL UNIFORMITY VS REVIVAL VARIETY IN PACKAGING',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('ORIGINAL UNIFORMITY VS REVIVAL VARIETY IN PACKAGING',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_packaging('horizontal')
plot_packaging('vertical')

In [ ]:
import numpy as np

df = packaging_data.to_pandas() if hasattr(packaging_data, 'to_pandas') else packaging_data
df = df.copy()
df.columns = [col.lower() for col in df.columns]

shape_counts = df.groupby(['era', 'pkg_shape']).size().reset_index(name='count')
totals = df.groupby('era').size().reset_index(name='total')
shape_counts = shape_counts.merge(totals, on='era')
shape_counts['pct'] = (shape_counts['count'] / shape_counts['total'] * 100).round(1)

pivot = shape_counts.pivot(index='pkg_shape', columns='era', values='pct').fillna(0)
pivot = pivot.reindex(columns=['original_launch', 'revival'])
pivot = pivot.sort_values('original_launch', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

y = np.arange(len(pivot))
bar_height = 0.35

bars_orig = ax.barh(y + bar_height/2, pivot['original_launch'], bar_height,
                    label='2013 Collection', color='#1a1a1a', edgecolor='#333', linewidth=0.5)
bars_revival = ax.barh(y - bar_height/2, pivot['revival'], bar_height,
                       label='2026 Revival', color='#B57EDC', edgecolor='#333', linewidth=0.5)

for bar in bars_orig:
    width = bar.get_width()
    if width > 0:
        ax.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.0f}%',
                va='center', fontsize=9, fontweight='normal', color='#1a1a1a')

for bar in bars_revival:
    width = bar.get_width()
    if width > 0:
        ax.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.0f}%',
                va='center', fontsize=9, fontweight='normal', color='#7B2D8B')

ax.set_yticks(y)
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_xlabel('% of products', fontsize=11, fontweight='normal')
ax.set_title('Package shape distribution', fontsize=13, fontweight='medium')
ax.legend(loc='lower right', fontsize=10, frameon=False)
ax.set_xlim(0, float(pivot.values.max()) + 12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.2)
ax.tick_params(labelsize=9)

orig_dominant = pivot['original_launch'].idxmax()
orig_pct = pivot['original_launch'].max()
revival_max = pivot['revival'].max()
ax.text(0.98, 0.02,
        f'Original: {orig_pct:.0f}% concentrated in \'{orig_dominant}\'\nRevival: max shape is only {revival_max:.0f}% -- spread across {(pivot["revival"] > 0).sum()} shapes',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        style='italic', color='#555', fontweight='normal')

fig.suptitle('REVIVAL SPREADS ACROSS MORE PACKAGE SHAPES',
             fontsize=18, fontweight='medium', y=0.98)
plt.tight_layout()
plt.show()